# Baseline IoU accuracy on Tumour Mask

In [ ]:
import pandas as pd
import numpy as np
from PIL import Image
from datasets import load_dataset
from sklearn.model_selection import train_test_split

In [ ]:
ds = load_dataset("chehablab/UCSF_PDGM", split="train", keep_in_memory=True)
df = pd.DataFrame(ds, columns=["volume_id", "slice_id", "tumor_mask", "is_tumorous"])
df = df[df["is_tumorous"] == True]

In [ ]:
patient_ids = np.asarray(df["volume_id"].unique(), dtype=object) # Splits by patient
train_ids, test_ids = train_test_split(patient_ids, test_size=0.3, random_state=1606009)
train_df = df[df["volume_id"].isin(train_ids)].reset_index(drop=True)
test_df = df[df["volume_id"].isin(test_ids)].reset_index(drop=True)
train_masks = df["tumor_mask"]
train_masks = [np.array(mask) for mask in train_masks]
test_masks = [np.array(mask) for mask in test_masks]


## Compute Average Mask

In [ ]:
stacked = np.stack(train_masks, axis=0)
pixel_freq = stacked.mean(axis=0)
avg_mask = (pixel_freq >= 0.5).astype(np.uint8)

## Compute IoU

In [ ]:
def compute_iou(avg_mask, test_mask):
    pred_mask = avg_mask.astype(bool)
    true_mask = test_mask.astype(bool)
    intersection = np.logical_and(pred_mask, true_mask).sum()
    union = np.logical_or(pred_mask, true_mask).sum()
    if union == 0:
        return 1.0
    return intersection / union

ious = [compute_iou(avg_mask, test_mask) for test_mask in test_masks]
iou = np.mean(ious)
print(f"Average mask baseline IoU: {iou:.4f}")

Average mask baseline IoU: 0.2732
